<a href="https://colab.research.google.com/github/yulialee9732/COMP577-Deep-learning/blob/main/COMP577_HW2_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 2. Deep learning

In this homework you will play with deep neural networks in PyTorch: write pieces of code for training/testing routine, train a simple baseline network, design your own architecture and pick parameters to beat the baseline, fine-tune two pre-trained deep neural nets (a ResNet and a Vision Transformer), create a simple region proposal net and compare it with a modern object detector, and finally generate images with pre-trained text-to-image diffusion models.

The homework is graded out of 50 points:
1. (5 points) implement testing loop and measure accuracy;
2. (10 points) design your architecture and beat the baseline accuracy on the CIFAR-10 dataset (better accuracy -> more points);
3. (5 points) fine-tune ResNet-50 on the Dogs-vs-Cats dataset;
4. (5 points) fine-tune ViT-Small on the Dogs-vs-Cats dataset and compare it with ResNet-50;
5. (5 points) use your model to create a simple region proposal net;
6. (5 points) run a YOLO detector and compare it with your region proposals;
7. (10 points) text-to-image generation: study the effect of the noise, the number of sampling steps, and the choice of model;
8. (15 points) discussion: answer seven short **Discussion** prompts inside this notebook (a few sentences each, about your experimental results).

This is a coding assignment: there is no separate report. Write your answers to the Discussion prompts directly in the corresponding markdown cells.

### Submission format:

Notebook with your work (shared link or `.ipynb`), **with all outputs saved**.

You can save a copy of this notebook in Colab and just submit the link, or download the `.ipynb` notebook and submit that way. Please make sure that your link is properly shared (it's best to pick "**everyone with the link can comment**"), or make sure that your `.ipynb` has all your outputs (there is an option to download just the code/text-only lightweight version). The generated images in Section 5 must be visible in the saved outputs.

Before you submit, make sure your notebook runs from start to end through **"Runtime -> Restart and Run all"**. If it doesn't, this likely means you executed cells out-of-order, or created variables with inconsistent names (e.g. one of the cells was using an output of a temporary variable from a cell that you deleted). Your results can also differ if you execute the training loop several times without resetting the network (making more epochs, which may result in a better result, but may also lead to overfitting).

# Notes on using Google Colab

**Important: make sure your work is saved to your Google Drive or Github before starting**

`File --> Save a copy...`. See the [docs](https://www.tutorialspoint.com/google_colab/google_colab_saving_work.htm).

In the top right corner you can see the available resources (RAM, Disk, etc.). Click on that button to expand the tab. By default you may get a CPU-only machine allocated. You can request an instance with a GPU through `Runtime -> Change runtime type` in the file menu bar. Public GPUs may not always be available. CPU is sufficient for Sections 1-4, as you don't need to train the models from scratch on large datasets (fine-tuning in Section 3 is much faster on a GPU though). **Section 5 (diffusion models) requires a GPU runtime.** Free GPUs are not always available and sessions have a time limit, so plan ahead: run Sections 1-4 first, save the notebook, and run Section 5 in a separate session if needed.

In [ ]:
# Install the extra packages used in Sections 3-5.
# timm: Vision Transformers | ultralytics: YOLO | diffusers: text-to-image models
!pip install -q timm ultralytics diffusers transformers accelerate

In [ ]:
# Import required packages
import os
import time
import zipfile
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split

import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet50, ResNet50_Weights

In [ ]:
# We will use this function to put objects (networks, inputs, labels)
# to GPU if one is available
def to_device(obj):
    if torch.cuda.is_available():
        obj = obj.to("cuda")
    return obj

# 1. CIFAR-10 image classification

## 1.1 CIFAR-10 data


The [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html) is a well-known image classification dataset that contains 60,000 32x32 color images in 10 classes, with 6,000 images per class. The goal of image classification is to correctly classify each image into its corresponding class label.

Achieving high accuracy on CIFAR-10 can be challenging, as the images are relatively small and contain many details, making it difficult to distinguish between some of the classes.

We will create a simple convolutional neural network and train it to achieve good baseline accuracy.

But first, let's look at some of the CIFAR-10 data samples. It's always beneficial to understand your data before attempting to solve the problem!

Luckily, PyTorch's `torchvision` package has a native CIFAR-10 dataset implementation, so creating the dataset can be as simple as `data = torchvision.datasets.CIFAR10(root='./data', download=True)`. However, we will need a couple more things to make it handy for network training.

First, define a preprocessing function that is used to transform the images in the dataset. In this case, the `ToTensor` transform is used to convert the images to PyTorch tensors, and the `Normalize` transform is used to normalize the tensor values to some range to facilitate the network convergence.

Second, remember to use different datasets for training and testing. `CIFAR10` handles that through the `train` flag already, but normally you would split the dataset into two parts to test on unseen images.


Finally, create data loaders that iterate over the train and test sets. The `batch_size` and `shuffle` arguments specify the iteration batch size and whether the data should be shuffled between epochs, and the `num_workers` argument specifies the number of subprocesses to use for data loading.

In [ ]:
preprocess = transforms.Compose(
    [transforms.ToTensor(), # convert the data to PyTorch tensors
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]) # normalize the data

cifar_train = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=preprocess)
cifar_test = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=preprocess)

batch_size = 32 # number of samples processed before the model is updated
cifar_train_loader = torch.utils.data.DataLoader(
    cifar_train, batch_size=batch_size, shuffle=True, num_workers=2)
cifar_test_loader = torch.utils.data.DataLoader(
    cifar_test, batch_size=batch_size, shuffle=False, num_workers=2)

In order to show the images in the batch, we now need to un-normalize them since the original images were normalized during the dataset loading process. Let's plot the first few images from the batch and print their labels.

In [ ]:
def show_images(img, mean, std):
    # Un-normalize the image and plot it
    mean = torch.tensor(mean, device=img.device)
    std = torch.tensor(std, device=img.device)
    img = img.permute(1, 2, 0)
    img = img * std + mean
    plt.imshow(img.numpy())


# fetch a batch from the train dataset
dataiter = iter(cifar_train_loader)
images, labels = next(dataiter)
nimages = min(batch_size, 4)
norm = preprocess.transforms[-1]
show_images(torchvision.utils.make_grid(images[:nimages]), norm.mean, norm.std)
# print labels
print(' '.join('%5s' % cifar_train.classes[labels[j]] for j in range(nimages)))

## 1.2 Baseline network

The SimpleNet class below inherits from the nn.Module class, which is a base class for all neural network modules in PyTorch. The __init__ method initializes the various layers of the neural network, while the forward method defines the forward pass through the network. Notice that you don't have to implement the backward pass when using superpositions of standard functions and layers, since the corresponding backward pass is already implemented for these standard operations, and the backward pass for your full model is simply derived from those using the chain rule.

Our network consists of two convolutional layers (conv1 and conv2) followed by two fully connected layers (fc1 and fc2), and a final output layer (fc3) with 10 output units (one for each class in the CIFAR-10 dataset). It also has a 2x down-sampling `pool` layer (applied after both conv1 and conv2), and ReLU nonlinearities between the layers.

The input to the network is a batch of 32x32 color images (3 channels for RGB) and the output is a tensor of shape (batch_size, 10) representing the predicted class scores for each input image in the batch.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_size=32, num_classes=10):
        # define the layers of the network
        super(SimpleNet, self).__init__()
        # 3 input channels: RGB
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.fc1 = nn.Linear(in_features=16 * 5 * 5, out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=84)
        # num_classes output channels
        self.fc3 = nn.Linear(in_features=84, out_features=num_classes)

    def forward(self, x):
        # Remember to use nonlinearities after hidden linear layers (conv, fc)
        # Convolutional layers
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # Reshape as a vector
        x = x.view(-1, self.fc1.in_features)
        # Fully connected (fc) layers
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        # Note that the output layer does not need a nonlinearity.
        # as it models log-probability
        x = self.fc3(x)
        return x


model_simple = SimpleNet()
model_simple = to_device(model_simple)

## 1.3 Training and testing the network

In order to train the network, we are still missing a couple things:
1. A loss criterion: for most classification problems, cross-entropy is used to define loss. Luckily, we know ground truth labels for our training dataset, and our network already models log-probability output with its final fully connected layer.

2. An optimizer: we will use SGD with momentum as our base optimizer. You will have to tune its parameters later, and feel free to explore other optimizers available in the `optim` package!

The training code runs over the dataset `nepoch` number of times (epochs).

## TODO #1 (5 points): implement the testing loop

While training code is provided below, you have to implement the testing routine. It's very similar to training, even simpler, as there is no need to make a call into the optimization step! You also have to loop through the data and get network outputs. Instead of computing the loss, let's measure the accuracy of the predictions:
$$\text{accuracy} = \dfrac{\text{number_of_correct_predictions}}{\text{total_samples}} * 100 \%$$

Use `torch.max()` to get the most likely label for every image from its per-class scores.

In [ ]:
def test(model, loader):
    # Testing loop
    # Make sure to set the network to the testing mode
    # torch.no_grad() makes sure extra resources are not
    # allocated during the network run as backward pass is not needed
    model.eval()
    with torch.no_grad():

        ###################################################
        # TODO: write your code here
        # Return the accuracy
        # Hint: look at the training code below for reference!
        ###################################################

        assert False, "Implement the testing loop! Remove this assert statement."
        accuracy = 0.0
    return accuracy


def train_and_test(model, nepochs,
                   train_loader, test_loader,
                   optimizer=optim.SGD,
                   optimizer_params=None):

    criterion = nn.CrossEntropyLoss()
    # Only pass parameters that require gradients: this matters for fine-tuning
    # frozen models in Section 3.
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optimizer(params, **optimizer_params)

    for epoch in range(nepochs):
        print("Epoch {}/{}".format(epoch + 1, nepochs))
        t0 = time.time()

        # Training loop
        # Some networks have different behavior for training and testing,
        # for example when BatchNorm or Dropout is used.
        # Make sure to set the network to the training mode.
        model.train()
        for inputs, labels in tqdm(train_loader):
            inputs = to_device(inputs)
            labels = to_device(labels)

            # Zero out the gradients before every iteration!
            optimizer.zero_grad()

            # forward + backward + optimizer step
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        train_time = time.time() - t0
        accuracy = test(model, test_loader)
        print('Epoch {}, Eval Accuracy: {:.2f}% (training time {:.0f}s)'.format(epoch+1, accuracy, train_time))

In [ ]:
optimizer_params_default = {
    "lr": 2e-3,
    "momentum": 0.9,
}
train_and_test(model_simple, 2, cifar_train_loader, cifar_test_loader,
               optimizer_params=optimizer_params_default)

Now let's look at some predictions


In [ ]:
dataiter = iter(cifar_test_loader)
images, labels = next(dataiter)

# print images
nimages = min(batch_size, 8)
norm = cifar_test.transform.transforms[-1]
show_images(torchvision.utils.make_grid(images[:nimages]), norm.mean, norm.std)
print('GroundTruth: ', ' '.join('%5s' % cifar_test.classes[labels[j]] for j in range(nimages)))

###################################################
# TODO: write your code here
# Print out the network predictions
###################################################

assert False, "Print out the predicted class labels."

# 2. Design your own architecture

You can start exploring the space of architectures by tuning the hyperparameters of our baseline network first. Hyperparameters are variables that are not learned from the data, but rather set before the training process begins. They affect how the model is trained and can significantly impact its performance. Those are, for example, learning rate, number of layers (depth), number of neurons per layer (width).

Choosing appropriate hyperparameters is crucial for achieving good performance, and is often done through a combination of trial and error and intuition based on prior experience.


Let's reserve a small validation set for parameter tuning out of our testing data, and test our final model on the test set, minus the validation set.


In [ ]:
# Split the test set into val and test_minus_val (20% and 80%, respectively)
# Use manual seed for results repeatability.
generator = torch.Generator().manual_seed(42)
cifar_val, cifar_test_minus_val = random_split(cifar_test, [0.2, 0.8],
                                               generator=generator)

cifar_val_loader = torch.utils.data.DataLoader(
    cifar_val, batch_size=batch_size, shuffle=False, num_workers=2)
cifar_test_minus_val_loader = torch.utils.data.DataLoader(
    cifar_test_minus_val, batch_size=batch_size, shuffle=False, num_workers=2)


Try tuning your parameters, and then modify the architecture, adding layers, normalization techniques, etc. from Pytorch's `nn` package.

## TODO #2 (10 points): Implement a stronger network and achieve high accuracy on CIFAR-10

In [ ]:
class CustomNet(nn.Module):
    def __init__(self):
        ###################################################
        # TODO: write your code here
        # Initialize network layers
        ###################################################
        raise NotImplementedError("Implement the initialization")

    def forward(self, x):
        ###################################################
        # TODO: write your code here
        # Implement the forward pass
        ###################################################
        raise NotImplementedError("Implement the initialization")

model_custom = CustomNet()
model_custom = to_device(model_custom)
optimizer_params_custom = {}
train_and_test(model_custom, 2, cifar_train_loader, cifar_val_loader,
               optimizer_params=optimizer_params_custom)

After you are done tuning the architecture structure and the hyperparameters, report your results with both CustomNet and SimpleNet on the `train_minus_val` split below. Discuss your resulting architecture in Discussion 1 below.

In [ ]:
accuracy_simple = test(model_simple, cifar_test_minus_val_loader)
print('Baseline model accuracy (test_minus_val): {:.2f}%'.format(accuracy_simple))

accuracy_custom = test(model_custom, cifar_test_minus_val_loader)
print('Custom model accuracy (test_minus_val): {:.2f}%'.format(accuracy_custom))

### Discussion 1
Describe your final architecture and training setup (layers, normalization, optimizer, learning rate, epochs). Which change gave you the largest improvement?

*Your answer here.*

# 3. Dogs-vs-Cats and model tuning

Let's now explore some images with higher resolution than those of 32x32 from CIFAR. One of the popular classification datasets is ["Dogs vs Cats"](https://www.kaggle.com/c/dogs-vs-cats/data). We will use a smaller version of the dataset here, picking 2000 images subset for training. Let's download and extract the archive to the /tmp folder.

In [ ]:

# Download the same 2,000 training and 1,000 validation images
# from an alternative host of the original ZIP.
import urllib.request

dataset_url = "https://storage.googleapis.com/tensorflow-1-public/course2/cats_and_dogs_filtered.zip"
local_zip = "/tmp/cats_and_dogs_filtered.zip"
urllib.request.urlretrieve(dataset_url, local_zip)

if not zipfile.is_zipfile(local_zip):
    raise RuntimeError(f"Download is not a valid ZIP file: {local_zip}")

with zipfile.ZipFile(local_zip) as zip_ref:
    bad_file = zip_ref.testzip()
    if bad_file is not None:
        raise RuntimeError(f"Corrupt image archive: {bad_file}")
    zip_ref.extractall("/tmp")

train_data = ImageFolder("/tmp/cats_and_dogs_filtered/train")
validation_data = ImageFolder("/tmp/cats_and_dogs_filtered/validation")

assert train_data.classes == validation_data.classes == ["cats", "dogs"]
assert len(train_data) == 2000 and len(validation_data) == 1000

print(f"Loaded {len(train_data)} training and {len(validation_data)} validation images.")
train_data[0][0]


Let's see how our simple (or your custom) model trained on CIFAR does on these images. For this one, mine predicted a `truck` class as a top prediction with 41.3% confidence. If yours predicted a cat: great! But how does it do on other images? Try a few more changing the `sample_idx` below.

In [ ]:
def get_prediction(img, model, preprocess=None, class_names=None):
    batch = preprocess(img) if preprocess is not None else img
    batch = batch.unsqueeze(0)
    batch = to_device(batch)
    model.eval()
    with torch.no_grad():
        prediction = model(batch).squeeze(0).softmax(0)
    class_id = prediction.argmax().item()
    score = prediction[class_id].item()
    category_name = class_names[class_id] if class_names is not None else class_id
    return category_name, score


def get_class_score(img, model, preprocess, class_id):
    """Probability of a *specific* class (not the winning one) for a single image."""
    batch = to_device(preprocess(img).unsqueeze(0))
    model.eval()
    with torch.no_grad():
        return model(batch).squeeze(0).softmax(0)[class_id].item()

sample_idx = 0
img = train_data[sample_idx][0]
img = img.resize((32, 32))
category_name, score = get_prediction(
    img, model_simple, cifar_train.transforms.transform, cifar_train.classes)
print(f"{category_name}: {100 * score:.1f}%")
img

## Fine tuning

Fine-tuning is the process of taking a strong pre-trained model and adapting it to a new dataset by training it further. Our pre-trained model is a 50-layer ResNet architecture, trained on the large ImageNet dataset to distinguish between 1000 classes.

Fine-tuning involves freezing the weights of the lower layers in the pre-trained model, which are responsible for low-level features such as edges and textures, and training the upper layers on the new dataset. The weights in the upper layers are initialized randomly and then updated through backpropagation during training.

We want to leverage the features of the ResNet and consider an extreme case, when all the layers are frozen, except for the last linear (fully connected) layer.

In [ ]:
# Imagenet accuracy: 80.858%
weights=ResNet50_Weights.DEFAULT
model_resnet = resnet50(weights=weights)
model_resnet = to_device(model_resnet)

ImageNet doesn't have a simple `cat` class: it rather has several classes such as `Egyptian cat`, `tabby cat`, etc. Assigning a cat to a wrong breed is considered the same as assigning a completely unrelated label. For example `sample_idx=3` is assigned to a `lynx` label (reasonable?), while `sample_idx=40` is assigned a `chihuahua` label as cats don't fall under the ImageNet specific breeds.

In [ ]:
sample_idx = 3
img = train_data[sample_idx][0]
category_name, score = get_prediction(img, model_resnet, weights.transforms(),
                                      weights.meta["categories"])
print(f"{category_name}: {100 * score:.1f}%")
img

## TODO #3 (5 points): Fine-tune the ResNet

Let's fine-tune the pre-trained ResNet. This involves the following steps:
1. freeze the pretrained layers;
2. modify the output to match the Dogs-vs-Cats class: you can simply change the `model_resnet.fc` output layer: there's no need to write new classes (e.g. if you wanted the last layer to be a convolution, you could re-define it as `model_resnet.fc = nn.Conv2D(3, 4, 5)`);
3. run `train_and_test` with appropriate arguments.

Can you achieve high accuracy now? Record the test accuracy and the time per epoch: you will compare them with a Vision Transformer in the next part.

In [ ]:
# Dogs-vs-Cats train and test sets
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
catsdogs_train = ImageFolder("/tmp/cats_and_dogs_filtered/train", transform=train_transforms)
catsdogs_test = ImageFolder("/tmp/cats_and_dogs_filtered/validation", transform=test_transforms)
catsdogs_train_loader = torch.utils.data.DataLoader(
    catsdogs_train, batch_size=32, shuffle=True, num_workers=2)
catsdogs_test_loader = torch.utils.data.DataLoader(
    catsdogs_test, batch_size=32, shuffle=False, num_workers=2)

# Freeze the parameters
for param in model_resnet.parameters():
    param.requires_grad = False


###################################################
# TODO: write your code here
# Modify the ResNet, and fine-tune to Dogs-vs-Cats
###################################################

assert False, "Modify the last layer of ResNet and fine-tune!"

## Fine tuning a Vision Transformer

Convolutional networks are not the only option for a backbone. Vision Transformers (ViT) split the image into fixed-size patches (here 16x16 pixels), embed each patch as a token, and process the sequence of tokens with a standard Transformer encoder, the same architecture used in language models. A special `[CLS]` token summarizes the whole image and feeds the classification head.

We use `vit_small_patch16_224` from the `timm` library (about 22M parameters, pre-trained on ImageNet), which is comparable in size to ResNet-50 (about 25M parameters). Note that this model expects a **different input normalization** from the ResNet (mean and std of 0.5), so we build the transforms from the model's own data configuration instead of reusing the ResNet ones.

## TODO #4 (5 points): Fine-tune the ViT and compare with ResNet-50

Repeat the fine-tuning procedure with the ViT:
1. freeze the pretrained layers;
2. modify the output to match the Dogs-vs-Cats classes: change the `model_vit.head` layer (`model_vit.head.in_features` gives the input size);
3. run `train_and_test` with the same number of epochs as for the ResNet.

Then fill in the comparison table in Discussion 2.

In [ ]:
try:
    import timm
except ImportError:
    !pip install -q timm
    import timm

model_vit = timm.create_model("vit_small_patch16_224", pretrained=True)
model_vit = to_device(model_vit)

# Build the transforms that match this model's pre-training
vit_config = timm.data.resolve_data_config({}, model=model_vit)
print(vit_config)
vit_test_transforms = timm.data.create_transform(**vit_config, is_training=False)
vit_train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(vit_config["mean"], vit_config["std"])
])

catsdogs_train_vit = ImageFolder("/tmp/cats_and_dogs_filtered/train", transform=vit_train_transforms)
catsdogs_test_vit = ImageFolder("/tmp/cats_and_dogs_filtered/validation", transform=vit_test_transforms)
catsdogs_train_vit_loader = torch.utils.data.DataLoader(
    catsdogs_train_vit, batch_size=32, shuffle=True, num_workers=2)
catsdogs_test_vit_loader = torch.utils.data.DataLoader(
    catsdogs_test_vit, batch_size=32, shuffle=False, num_workers=2)

# Freeze the parameters
for param in model_vit.parameters():
    param.requires_grad = False

###################################################
# TODO: write your code here
# Modify the head of the ViT, and fine-tune to Dogs-vs-Cats
###################################################
assert False, "Modify the head of the ViT and fine-tune!"

In [ ]:
# Count trainable vs. total parameters for both models
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

for name, m in [("ResNet-50", model_resnet), ("ViT-Small", model_vit)]:
    total, trainable = count_params(m)
    print(f"{name}: {total/1e6:.1f}M total, {trainable/1e3:.1f}K trainable")

### Discussion 2
Fill in the table and answer the question.

| Model | Total params | Trainable params | Test accuracy | Time / epoch |
|-------|--------------|------------------|---------------|--------------|
| ResNet-50 (frozen + new fc) | | | | |
| ViT-Small (frozen + new head) | | | | |

Which backbone worked better here, and by how much (accuracy and time)? Name one plausible reason for the difference (think about pre-training data, the inductive biases of convolutions vs. attention, or the fixed-feature setting).

*Your answer here.*

## 4. Beyond image classification

So far we looked only at the image classification task, which seems like a very narrow (and easy?) problem. But in fact, image classification networks are used as a core (backbone) for most of the image recognition problems.

Our fine-tuned network distinguishes cats from dogs, but it also gives us its prediction confidence. Let's try to use this confidence to add (some) localization capabilities to the network.

Object localization, or object detection involves modifying the network architecture to output both a class label for the input image and a bounding box that specifies the location of the object in the image. We won't build a complete detector, but we will make a step towards creating a region proposal network that says how likely a selected region contains an object of interest.

Imagine that you know the size of the object you are looking for. For instance, in the image below, assume you know that you look for objects that occupy about 3% of the image that fit into a square of size $K \times K$. For the image below, that would mean that the square has a side of $K = 170$ pixels. Let's see what confidence our network has for a random patch of size $K \times K$ and for a patch that is centered on the dog. Note that we ask for the probability of the `dog` class specifically (`get_class_score`), not the probability of whichever class wins: a patch that is confidently classified as `cat` is *not* a good dog proposal.

This may sound counterintuitive from the first glance that this process has any meaning. Our network can only predict `cat` or `dog` class, and we are asking it to classify a patch of the sky or forest! However, the fact that the network doesn't see any cat or dog features in the patch will hopefully push it towards predicting the corresponding class with a lower confidence.

In [ ]:
# Load a high rest image containing a dog and display it.
!wget https://www.blossomsfield.co.uk/wp-content/uploads/2020/10/WhatsApp-Image-2020-10-06-at-12.43.20.jpeg \
      -O /tmp/localize.jpeg
img = Image.open("/tmp/localize.jpeg").convert("RGB")
width, height = img.size
width, height = 800, int(height * 800 / width)
img = img.resize((width, height))
img

In [ ]:
# Running our fine-tuned ResNet on a background patch.
dog_class_id = catsdogs_test.classes.index("dogs")

left = 35
upper = 740
size = 170
box_random = (left, upper, left + size, upper + size)
crop_random = img.crop(box_random)
score_random = get_class_score(crop_random, model_resnet, test_transforms, dog_class_id)
print(f"P(dog) = {100 * score_random:.1f}%")
crop_random

In [ ]:
# Running our fine-tuned ResNet on a dog patch.
left = 290
upper = 520
size = 170
box_center = (left, upper, left + size, upper + size)
crop_center = img.crop(box_center)
score_center = get_class_score(crop_center, model_resnet, test_transforms, dog_class_id)
print(f"P(dog) = {100 * score_center:.1f}%")
crop_center

Use the function below to draw a set of patches on top of the image.

In [ ]:
def draw(boxes, scores, color=(255, 0, 0)):
    width, height = img.size
    nboxes = len(boxes)
    pad = 10
    im2show = img.copy()
    img_draw = ImageDraw.Draw(im2show)
    for i, box in enumerate(boxes):
        score = scores[i]
        img_draw.rectangle(box, outline=color, width=5)

    return im2show

boxes = [box_center, box_random]
scores = [score_center, score_random]
im2show = draw(boxes, scores, color="blue")
im2show

## TODO #5 (5 points): Implement the sliding window and visualize the most and the least confident predictions

Let's produce a set of candidates in a sliding window fashion. That is, place a window of size $K \times K$ into the top left corner, and evaluate the score of the corresponding crop to contain class `dog`. Now slide this window across all possible locations with step $s$ (i.e. evaluate a crop of size $K \times K$ with the top left corner coordinate $(0, s)$ next). Record all window positions and the corresponding scores for the `dog` class. Then sort them and plot top-20 and bottom-20 scoring ones. Pick $s$ yourself, and $K = 170$. Use the probability of the `dog` class for every crop (`get_class_score`, or batch the crops and take `softmax(...)[:, dog_class_id]`), **not** the confidence of the predicted class.

In [ ]:
###################################################
# TODO: write your code here
# Implement the sliding window approach.
# Store the results in two lists, `boxes` and `scores`, sorted by score in
# decreasing order. Boxes are (left, upper, right, lower) tuples in the
# coordinates of `img`.
###################################################

In [ ]:
# Visualize the top-20 scoring boxes
high_scoring_img = draw(boxes[:20], scores[:20], color="green")
high_scoring_img

In [ ]:
# Visualize the bottom-20 scoring boxes
low_scoring_img = draw(boxes[-20:], scores[-20:], color="red")
low_scoring_img

Discuss your results: where did the top-20 scoring boxes land? How about the bottom-20 ones? Do any of the low scoring boxes still contain any pixels belonging to the dog? Why might that be (i.e. why a set of pixels belonging to the dog may score lower than a corner box with just the sky or just the grass)?

### Discussion 3
*Your answer here.*

## 4.2 Comparing with a modern object detector: YOLO

Our sliding window is a crude version of what detectors did a decade ago. Modern detectors like **YOLO** ("You Only Look Once") run the backbone once on the whole image and predict boxes and classes for all objects in a single forward pass, at many scales and aspect ratios, in a fraction of a second.

Let's run a pre-trained YOLO model (YOLOv8s, trained on the COCO dataset, which includes a `dog` class) on the same image and compare it with our region proposals.

## TODO #6 (5 points): Run YOLO and compare it with your region proposals

1. Run YOLOv8s on the image (done for you below) and draw its `dog` detection(s) on the image together with your top-1 sliding-window box.
2. Implement `iou()` (intersection over union) and compute the IoU between YOLO's highest-confidence `dog` box (or its highest-confidence box overall, if it did not call the dog a `dog`) and (a) your top-1 sliding-window box, (b) the best IoU among your top-20 boxes.
3. Time both approaches: the wall-clock time of a warmed-up run of your sliding window (all crops, model already on the device) vs. a warmed-up YOLO call on the full image (done for you below). Report both numbers together with how many crops the sliding window evaluated (and in how many batched forward calls) versus the single full-image pass of YOLO.

In [ ]:
try:
    from ultralytics import YOLO
except ImportError:
    !pip install -q ultralytics
    from ultralytics import YOLO

# yolov8s (~11M params) is a good trade-off; yolov8n often mislabels this shaggy
# dog on a hillside as 'sheep'. Try 'yolov8m.pt' if you want an even stronger model.
model_yolo = YOLO("yolov8s.pt")  # downloads the weights on first use
model_yolo(img, verbose=False)   # warm-up run (model load, CUDA init) - not timed

t0 = time.time()
results = model_yolo(img, verbose=False, conf=0.1)  # runs on the 800px-wide PIL image
yolo_time = time.time() - t0
print(f"YOLO inference time: {yolo_time:.2f}s")

# Collect boxes in (left, upper, right, lower) format with class names
yolo_boxes, yolo_scores, yolo_names = [], [], []
for r in results:
    for b in r.boxes:
        yolo_boxes.append(tuple(b.xyxy[0].tolist()))
        yolo_scores.append(float(b.conf[0]))
        yolo_names.append(r.names[int(b.cls[0])])

for box, s, n in zip(yolo_boxes, yolo_scores, yolo_names):
    print(f"{n:10s} {s:.2f} {tuple(int(v) for v in box)}")

# Draw YOLO detections (blue) and your top-1 sliding-window box (green)
im2show = draw(yolo_boxes, yolo_scores, color="blue")
ImageDraw.Draw(im2show).rectangle(boxes[0], outline="green", width=5)
im2show

In [ ]:
def iou(box_a, box_b):
    ###################################################
    # TODO: write your code here
    # Boxes are (left, upper, right, lower). Return intersection over union.
    ###################################################
    raise NotImplementedError("Implement IoU.")

###################################################
# TODO: write your code here
# 1. Find YOLO's highest-confidence 'dog' box. If YOLO did not label the dog
#    as 'dog' (COCO models sometimes say 'sheep' or 'cow' here), use its
#    highest-confidence box instead and mention this in Discussion 4.
# 2. Print the IoU with your top-1 sliding-window box.
# 3. Print the best IoU among your top-20 sliding-window boxes.
###################################################

### Discussion 4
How well does your sliding-window proposal agree with YOLO (report the IoUs and the timings)? What drawbacks does the sliding-window approach of creating region proposals have, and which of them does a detector like YOLO address (think about object size, aspect ratio, speed, and what happens when there are several objects)? What extra steps would you need to turn your region proposal net into a real detector?

*Your answer here.*

# 5. Generative models: text-to-image diffusion

**This section requires a GPU runtime** (`Runtime -> Change runtime type -> T4 GPU`). Each image takes a few seconds on a T4; the whole section should take about 10-15 minutes. If your session runs out, re-run only this section: it does not depend on the earlier ones.

So far our networks took an image as input and produced a label or a box. Generative models go the other way: they produce an image. Diffusion models generate an image by starting from pure Gaussian noise and iteratively denoising it, guided by a text prompt. Three things affect the result:

* **The initial noise** (set by the random seed). The same prompt with different seeds gives different images.
* **The number of denoising steps** (e.g. with the DDIM sampler). Fewer steps are faster but may lower the quality.
* **The model itself.** Models differ in size, training data, and training objective.

We use the `diffusers` library and two small models that fit on a free Colab GPU in half precision:

* `stable-diffusion-v1-5/stable-diffusion-v1-5` (Stable Diffusion v1.5, about 0.9B parameters, 512x512 images) with a DDIM scheduler;
* `stabilityai/sd-turbo` (SD-Turbo, a distilled model that produces an image in 1-4 steps without classifier-free guidance).

**Content note:** pick neutral prompts (objects, animals, landscapes, scenes). Prompts that name real people or contain sensitive content will not be graded.

In [ ]:
try:
    import diffusers
except ImportError:
    !pip install -q diffusers accelerate
from diffusers import StableDiffusionPipeline, AutoPipelineForText2Image, DDIMScheduler

assert torch.cuda.is_available(), "Section 5 needs a GPU runtime."
device = "cuda"

# Model A: Stable Diffusion v1.5 with a DDIM sampler
pipe_sd = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,  # keep it light; you are responsible for your prompts
)
pipe_sd.scheduler = DDIMScheduler.from_config(pipe_sd.scheduler.config)
pipe_sd = pipe_sd.to(device)
pipe_sd.enable_attention_slicing()  # lower memory use
pipe_sd.set_progress_bar_config(disable=True)

In [ ]:
def generate(pipe, prompt, seed, num_steps=25, guidance_scale=7.5, size=512):
    '''Generate one image with a fixed seed (i.e. fixed initial noise).'''
    g = torch.Generator(device=device).manual_seed(seed)
    out = pipe(prompt, num_inference_steps=num_steps, guidance_scale=guidance_scale,
               height=size, width=size, generator=g)
    return out.images[0]

def show_grid(images, titles=None, ncols=None, figsize_per_img=3):
    '''Display a list of PIL images in a grid.'''
    n = len(images)
    ncols = ncols or n
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per_img * ncols, figsize_per_img * nrows))
    axes = np.array(axes).reshape(-1)
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < n:
            ax.imshow(images[i])
            if titles is not None:
                ax.set_title(titles[i], fontsize=9)
    plt.tight_layout()
    plt.show()

# Quick test
show_grid([generate(pipe_sd, "a photo of a corgi wearing sunglasses on a beach", seed=0)])

## TODO #7 (10 points): noise, steps, and models

Choose **three text prompts** of your own. Make them reasonably specific (an object, an attribute, a setting; e.g. *"a red bicycle leaning against a brick wall, sunny day"*), and make at least one of them a bit unusual so that the models have to compose concepts they may not have seen together.

**(a) Varying the noise (3 points).** For each prompt, generate **4 images with SD v1.5** using seeds `0, 1, 2, 3` (fixed 25 DDIM steps). Show the 3x4 grid.

**(b) Varying the number of steps (3 points).** Pick one prompt and one seed. Generate images with **5, 10, 25, and 50** DDIM steps and show them side by side. Record the generation time for each.

**(c) Comparing models (4 points).** Run the same three prompts with the same 4 seeds through **SD-Turbo** (use `num_steps=1` and `guidance_scale=0.0`, which is how this model is meant to be used; you can also try 2-4 steps). Show the 3x4 grid next to the SD v1.5 one.

Then answer Discussion 5-7. Judge the results visually along two axes: **prompt fidelity** (does the image contain what the prompt asks for, with the right attributes and relations?) and **diversity** (how different are the images for different seeds?).

In [ ]:
prompts = [
    # TODO: replace with your own three prompts
    "PROMPT 1",
    "PROMPT 2",
    "PROMPT 3",
]
seeds = [0, 1, 2, 3]

###################################################
# TODO (a): write your code here
# Generate a 3 x 4 grid with SD v1.5: rows = prompts, columns = seeds
###################################################

In [ ]:
###################################################
# TODO (b): write your code here
# One prompt, one seed, num_steps in [5, 10, 25, 50]. Time each generation
# and put the step count and time in the image titles.
###################################################

In [ ]:
# Model B: SD-Turbo. Free the first pipeline first: the SD v1.5 images you
# need for the comparison are already stored in Python lists.
del pipe_sd
torch.cuda.empty_cache()
pipe_turbo = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16",
    safety_checker=None,
).to(device)
pipe_turbo.set_progress_bar_config(disable=True)

###################################################
# TODO (c): write your code here
# Same 3 prompts x 4 seeds with SD-Turbo (num_steps=1, guidance_scale=0.0)
###################################################

### Discussion 5 (noise)
For SD v1.5, how did changing the seed change the images? Was the diversity similar across your three prompts? Did any seed produce an image that does not match the prompt?

*Your answer here.*

### Discussion 6 (steps)
How did quality and generation time change from 5 to 50 DDIM steps? Around which number of steps did the improvement stop being noticeable?

*Your answer here.*

### Discussion 7 (models)
Compare SD v1.5 and SD-Turbo on prompt fidelity, diversity across seeds, and speed. Which model handled your "unusual" prompt better? Give one hypothesis for the differences, considering that SD-Turbo is distilled to generate in a single step.

*Your answer here.*

---
**Before submitting:** make sure all outputs (accuracies, box visualizations, and generated image grids) are saved in the notebook and all Discussion cells are filled in.